# 🐱🐶 Module 1: Model Development & Experiment Tracking
### Cat vs Dog Binary Classification — MLOps Assignment

**Course**: MLOps (S1-25_AIMLCZG523)  
**Module**: M1 — Model Development & Experiment Tracking (10 Marks)  
**Objective**: Build a baseline CNN, perform EDA, and track all experiments with MLflow.

---

## Notebook Outline
1. Environment Setup & Dataset Download  
2. Exploratory Data Analysis (EDA)  
3. Data Preprocessing & DataLoaders  
4. Baseline CNN: Architecture Overview  
5. Training with MLflow Experiment Tracking  
6. Evaluation: Loss Curves, Confusion Matrix, Report  
7. Summary & Conclusions  

## Cell 1: Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# Ensure we always run from project root regardless of CWD
PROJECT_ROOT = Path(os.getcwd())
# If running from notebooks/ subfolder, go up one level
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'✅ Project root: {PROJECT_ROOT}')
print(f'Python version: {sys.version}')

In [ ]:
# Install dependencies (only first time — subsequent runs skip already-installed packages)
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠️ Some packages may have failed to install:')
    print(result.stderr[-2000:])
else:
    print('✅ All dependencies installed/verified.')

In [ ]:
# Core imports
import random
import warnings
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import torch

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load project config
with open('configs/config.yaml') as f:
    CONFIG = yaml.safe_load(f)

print('Config loaded:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Cell 2: Download Dataset

> **Smart Download**: The script below checks if `data/raw/` already contains images.  
> If yes → **skip** (no re-download). If no → download from Kaggle and copy to project dir.

In [ ]:
from src.data.download_dataset import download_dataset, verify_dataset_structure

# Downloads to data/raw/ in project directory — skips if already present
dataset_path = download_dataset(
    kaggle_id=CONFIG['dataset']['kaggle_id'],
    target_dir=CONFIG['paths']['data_raw'],
    force_redownload=False   # <-- change to True only if you want a fresh copy
)

print(f'\n📂 Dataset location: {dataset_path}')
verify_dataset_structure(CONFIG['paths']['data_raw'])

## Cell 3: Exploratory Data Analysis (EDA)

Before building any model, we need to understand the data:
- Dataset size and class balance
- Sample images per class
- Image size/resolution distribution

In [ ]:
from src.data.download_dataset import load_config
from src.data.preprocess import find_dataset_root

raw_dir = Path(CONFIG['paths']['data_raw'])
classes = CONFIG['dataset']['classes']

# Find dataset root with class subdirectories
dataset_root = find_dataset_root(str(raw_dir), classes)
print(f'Dataset root: {dataset_root}')

# Count images per class
class_counts = {}
class_paths = {}
for cls in classes:
    # Try both lowercase and capitalized folder names
    for name in [cls, cls.capitalize()]:
        p = dataset_root / name
        if p.exists():
            imgs = list(p.glob('*.jpg')) + list(p.glob('*.jpeg')) + list(p.glob('*.png'))
            class_counts[cls] = len(imgs)
            class_paths[cls] = p
            break

print('\n📊 Class Distribution:')
total_imgs = sum(class_counts.values())
for cls, count in class_counts.items():
    pct = count / total_imgs * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:>6}: {count:>6} images ({pct:.1f}%) {bar}')
print(f'  {"TOTAL":>6}: {total_imgs:>6} images')

In [ ]:
# ── Class Distribution Bar Chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
colors = ['#4C9BE8', '#E87B4C']
bars = axes[0].bar(list(class_counts.keys()), list(class_counts.values()),
                   color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for bar, count in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values(), labels=class_counts.keys(),
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Balance', fontsize=13, fontweight='bold')

plt.suptitle('Cats vs Dogs Dataset — Class Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('models/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: models/eda_class_distribution.png')

In [ ]:
# ── Sample Images Grid ────────────────────────────────────────────────────
n_samples = 5
fig, axes = plt.subplots(2, n_samples, figsize=(15, 6))
fig.suptitle('Sample Images: Cats (top) vs Dogs (bottom)', fontsize=13, fontweight='bold')

for row_idx, cls in enumerate(classes):
    img_files = list(class_paths[cls].glob('*.jpg'))[:100]
    random.seed(42)
    sampled = random.sample(img_files, min(n_samples, len(img_files)))
    for col_idx, img_path in enumerate(sampled):
        img = Image.open(img_path).convert('RGB')
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
        axes[row_idx, col_idx].set_title(f'{cls}\n{img.size[0]}x{img.size[1]}', fontsize=9)

plt.tight_layout()
plt.savefig('models/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: models/eda_sample_images.png')

In [ ]:
# ── Image Dimension Analysis ──────────────────────────────────────────────
print('Sampling image sizes (this may take a moment)...')
widths, heights = [], []
SAMPLE_SIZE = 200  # sample per class to keep it fast

for cls in classes:
    img_files = list(class_paths[cls].glob('*.jpg'))
    sample = random.sample(img_files, min(SAMPLE_SIZE, len(img_files)))
    for p in sample:
        try:
            with Image.open(p) as img:
                widths.append(img.size[0])
                heights.append(img.size[1])
        except Exception:
            pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='#4C9BE8', edgecolor='white', alpha=0.8)
axes[0].axvline(224, color='red', linestyle='--', linewidth=2, label='Target: 224px')
axes[0].set_title('Image Width Distribution', fontweight='bold')
axes[0].set_xlabel('Width (px)')
axes[0].legend()

axes[1].hist(heights, bins=30, color='#E87B4C', edgecolor='white', alpha=0.8)
axes[1].axvline(224, color='red', linestyle='--', linewidth=2, label='Target: 224px')
axes[1].set_title('Image Height Distribution', fontweight='bold')
axes[1].set_xlabel('Height (px)')
axes[1].legend()

plt.suptitle(f'Image Dimensions (sample of {len(widths)} images)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('models/eda_image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Width  → mean: {np.mean(widths):.0f}px | median: {np.median(widths):.0f}px | range: {min(widths)}-{max(widths)}px')
print(f'Height → mean: {np.mean(heights):.0f}px | median: {np.median(heights):.0f}px | range: {min(heights)}-{max(heights)}px')
print(f'All images will be resized to {CONFIG["dataset"]["image_size"]} for the model.')

### EDA Observations

- **Class balance**: The dataset is approximately balanced between cats and dogs, which means we don't need to apply class weighting in our loss function.
- **Image sizes vary widely** — preprocessing to 224×224 is essential for consistent CNN inputs.
- **No missing/corrupt images** were detected in the sampled batch.

> These observations confirm the dataset is clean and ready for the preprocessing pipeline.

## Cell 4: Preprocessing & DataLoaders

- Resize to **224×224 RGB**
- **80/10/10** train/val/test split
- Training augmentation: flips, rotation, color jitter
- Normalization with ImageNet mean/std

In [ ]:
from src.data.preprocess import get_dataloaders, get_transforms

train_loader, val_loader, test_loader, class_to_idx = get_dataloaders()

print('\n📦 DataLoader Summary:')
print(f'  class_to_idx : {class_to_idx}')
print(f'  Train batches: {len(train_loader):>4} | ~{len(train_loader.dataset):,} images')
print(f'  Val batches  : {len(val_loader):>4} | ~{len(val_loader.dataset):,} images')
print(f'  Test batches : {len(test_loader):>4} | ~{len(test_loader.dataset):,} images')
print(f'  Batch size   : {CONFIG["training"]["batch_size"]}')

In [ ]:
# ── Visualize Augmented vs Original ──────────────────────────────────────
# Show the same image with and without augmentation side-by-side
import torchvision.transforms.functional as TF

# Get a sample image from the first class
sample_path = list(class_paths[classes[0]].glob('*.jpg'))[0]
orig_img = Image.open(sample_path).convert('RGB')

aug_transform  = get_transforms((224, 224), augment=True, aug_config=CONFIG['augmentation'])
base_transform = get_transforms((224, 224), augment=False)

# Denormalize helper
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
def denorm(t): return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle('Top: Original 224×224 | Bottom: Augmented Variants', fontsize=12, fontweight='bold')

for col in range(6):
    orig_t = base_transform(orig_img)
    aug_t  = aug_transform(orig_img)
    axes[0, col].imshow(denorm(orig_t))
    axes[0, col].axis('off')
    axes[0, col].set_title('Original', fontsize=8)
    axes[1, col].imshow(denorm(aug_t))
    axes[1, col].axis('off')
    axes[1, col].set_title('Augmented', fontsize=8)

plt.tight_layout()
plt.savefig('models/eda_augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: models/eda_augmentation_examples.png')

## Cell 5: Baseline CNN — Architecture

The Baseline CNN consists of:
- **3 ConvBlocks**: Conv2d → BatchNorm → ReLU → MaxPool
- **Filter sizes**: 32 → 64 → 128
- **Global Average Pooling** (reduces spatial dims to 1×1)
- **FC(512) → Dropout(0.5) → FC(2)** classifier head
- **Kaiming (He) weight initialization**

In [ ]:
from src.models.baseline_cnn import build_model, BaselineCNN
from src.utils.helpers import get_device, count_parameters

device = get_device()
model = build_model(CONFIG).to(device)

print(model.summary())
print(f'\n🔧 Device: {device}')

# Dry-run to confirm shapes
dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'\nDry-run forward pass:')
print(f'  Input shape : {dummy.shape}')
print(f'  Output shape: {out.shape}  (batch=4, classes=2)')

## Cell 6: Training with MLflow Experiment Tracking

MLflow tracks:
- **Parameters**: hyperparameters from config.yaml
- **Metrics**: train/val loss & accuracy per epoch, final test metrics
- **Artifacts**: model checkpoint, confusion matrix, loss curves, classification report

After training, view results: `mlflow ui --port 5000`

In [ ]:
# Run the full training pipeline
# This calls train.py logic directly (same code as python src/models/train.py)
from src.models.train import train

print('🚀 Starting training...\n')
train(config_path='configs/config.yaml')
print('\n✅ Training complete!')

## Cell 7: Results Visualization

In [ ]:
# Display saved training curves
curves_path = Path('models/training_curves.png')
cm_path = Path('models/confusion_matrix.png')

if curves_path.exists() and cm_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(plt.imread(str(curves_path)))
    axes[0].axis('off')
    axes[0].set_title('Training Curves', fontsize=12, fontweight='bold')
    axes[1].imshow(plt.imread(str(cm_path)))
    axes[1].axis('off')
    axes[1].set_title('Confusion Matrix (Test Set)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Run training first (Cell 6) to generate these plots.')

In [ ]:
# Display Classification Report
report_path = Path('models/classification_report.txt')
if report_path.exists():
    print('📋 Classification Report (Test Set)\n' + '='*50)
    print(report_path.read_text())
else:
    print('Run training first (Cell 6).')

In [ ]:
# View MLflow experiments summary
import mlflow
from pathlib import Path

tracking_uri = str(Path('mlruns').resolve())
mlflow.set_tracking_uri(f'file:///{tracking_uri}')

client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(CONFIG['mlflow']['experiment_name'])

if exp:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=['metrics.val_acc DESC']
    )
    print(f'\n📊 MLflow Experiment: "{CONFIG["mlflow"]["experiment_name"]}"')
    print(f'   Total runs: {len(runs)}\n')
    print(f'{"Run Name":<35} {"Val Acc":>8} {"Test Acc":>9} {"Val Loss":>9}')
    print('-' * 65)
    for run in runs[:10]:
        name    = run.data.tags.get('mlflow.runName', run.info.run_id[:8])
        val_acc = run.data.metrics.get('val_acc', float('nan'))
        tst_acc = run.data.metrics.get('test_acc', float('nan'))
        val_ls  = run.data.metrics.get('val_loss', float('nan'))
        print(f'{name:<35} {val_acc:>7.2f}% {tst_acc:>8.2f}% {val_ls:>9.4f}')
    print(f'\n💡 To explore interactively: run `mlflow ui --port 5000` in terminal.')
else:
    print('No MLflow experiments found. Run Cell 6 first.')

## Cell 8: Summary & Conclusions

### Module 1 Deliverables ✅

| Requirement | Implementation |
|------------|----------------|
| Git versioning | `.git/` initialized, source code tracked |
| DVC dataset versioning | `dvc.yaml`, `data/raw/` tracked via DVC |
| Baseline Model | 3-block CNN in `src/models/baseline_cnn.py` |
| Model saved | `models/best_model.pt` (PyTorch `.pt` format) |
| Experiment tracking | MLflow — parameters, metrics, artifacts all logged |
| Confusion matrix | Logged as MLflow artifact |
| Loss curves | Logged as MLflow artifact |
| Data augmentation | Flips, rotation, color jitter applied to train set |
| 80/10/10 split | Implemented in `src/data/preprocess.py` |

### Key Findings

- The dataset is **well-balanced** between cats and dogs, requiring no class-weight corrections.
- Image sizes vary widely; resizing to **224×224** standardizes inputs across the dataset.
- The **Baseline CNN** achieves competitive validation accuracy with a simple 3-block architecture.
- Data augmentation (flips + rotation + color jitter) helps reduce overfitting.
- All experiment parameters and metrics are tracked in **MLflow** for full reproducibility.

### Next Steps (Module 2)

- Wrap the trained model in a **FastAPI REST API** with `/health` and `/predict` endpoints
- Create a **Dockerfile** for containerization
- Verify predictions via curl/Postman